# SMB Data Exfiltration Detection: Three-Layer Pipeline

| Layer | Question | Method |
|---|---|---|
| 1 | Is this SMB flow abnormal? | Isolation Forest trained on **benign** CICIDS-2017 port-445 flows only |
| 2 | Is it data **leaving** the server? | Rule: >= 1 MB server-to-client and byte ratio >= 10 |
| 3 | Is that normal **for this host**? | Unknown host -> HIGH; above 3x own 10-min baseline -> MEDIUM; else LOW |

**Why unsupervised:** no public dataset contains labelled SMB exfiltration. The model learns normal SMB from real benign
traffic; exfiltration is evaluated on traffic generated with real attack tools in an isolated lab.

**Feature extraction:** CICIDS-2017 was built with CICFlowMeter. Lab captures are converted with CICFlowMeter (Python 0.5.0,
patched: wall-clock garbage collection disabled for offline pcaps; it otherwise split long flows and lost ~40% of bytes).
NIC offloading (TSO/GSO/GRO) was disabled on the capture host so packet sizes match real wire sizes (max 1514 B).

## 1. Train Layer 1 with leave-one-day-out validation

In [1]:
import glob, os, json, numpy as np, pandas as pd, joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)

BASE  = os.path.expanduser("~/Desktop/dns-exfiltration-ids")
files = glob.glob(os.path.expanduser("~/Desktop/**/*ISCX*.csv"), recursive=True)

# Exfiltration-focused features: volume, direction, rate, timing.
# Excluded on purpose: ports, TCP flag counts, Init_Win_bytes (OS / lab fingerprint).
FEATURES = [
    "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets",
    "Fwd Packet Length Mean", "Bwd Packet Length Max", "Bwd Packet Length Mean",
    "Flow Bytes/s", "Flow Packets/s", "Bwd Packets/s",
    "Flow IAT Mean", "Flow IAT Std", "Down/Up Ratio",
    "Average Packet Size", "Packet Length Std", "Active Mean", "Idle Mean",
]

parts = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df[df["Destination Port"] == 445].copy()
    df["day"] = os.path.basename(f).split("-")[0]
    parts.append(df)
data = pd.concat(parts, ignore_index=True)
data["Label"] = data["Label"].str.strip()
data[FEATURES] = data[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
data["Bwd_Fwd_Byte_Ratio"] = data["Total Length of Bwd Packets"] / (data["Total Length of Fwd Packets"] + 1)
FEATS = FEATURES + ["Bwd_Fwd_Byte_Ratio"]

benign = data[data["Label"] == "BENIGN"]
scan   = data[data["Label"] == "PortScan"]
print(f"CICIDS-2017 files: {len(files)} | port-445 BENIGN: {len(benign)} | PortScan: {len(scan)}")

prep = lambda d: np.log1p(d[FEATS].clip(lower=0).values)

def fit(train_df):
    sc = RobustScaler().fit(prep(train_df))
    m  = IsolationForest(n_estimators=300, contamination=0.01, random_state=42)
    m.fit(sc.transform(prep(train_df)))
    return m, sc

flag = lambda m, sc, d: (m.predict(sc.transform(prep(d))) == -1).mean() * 100

print("\n=== Leave-One-Day-Out ===")
print(f"{'Held-out day':12s} {'train':>6s} {'test':>6s} {'FP rate':>8s} {'PortScan flagged':>17s}")
fps = []
for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
    tr, te = benign[benign["day"] != day], benign[benign["day"] == day]
    m, sc = fit(tr)
    fp = flag(m, sc, te); fps.append(fp)
    print(f"{day:12s} {len(tr):6d} {len(te):6d} {fp:7.1f}% {flag(m, sc, scan):16.1f}%")
print(f"\nMean false positive rate on unseen days: {np.mean(fps):.1f}% (+/- {np.std(fps):.1f})")

model, scaler = fit(benign)   # final deployment model: all benign flows
print(f"Final model - PortScan flagged: {flag(model, scaler, scan):.1f}%  (low by design: scans move no data)")
joblib.dump(model,  f"{BASE}/smb_if_v2.pkl")
joblib.dump(scaler, f"{BASE}/smb_scaler_v2.pkl")
json.dump(FEATS, open(f"{BASE}/smb_features_v2.json", "w"), indent=1)

CICIDS-2017 files: 8 | port-445 BENIGN: 1932 | PortScan: 179

=== Leave-One-Day-Out ===
Held-out day  train   test  FP rate  PortScan flagged
Monday         1677    255     0.0%              8.4%
Tuesday        1560    372     2.7%             45.3%
Wednesday      1673    259     0.0%              0.6%
Thursday       1141    791     1.5%             44.7%
Friday         1677    255     0.0%              4.5%

Mean false positive rate on unseen days: 0.8% (+/- 1.1)
Final model - PortScan flagged: 0.6%  (low by design: scans move no data)


## 2. Scoring function (Layers 1 + 2)
Maps Python-CICFlowMeter columns to CICIDS names in the saved training order, and converts seconds to microseconds
(CICIDS time unit).

In [2]:
US, MB = 1e6, 1024 * 1024
cols = ["timestamp", "src_ip", "src_port", "flow_duration", "MB_out", "MB_in", "ratio", "if_score", "verdict"]

def score(path):
    raw = pd.read_csv(path).replace([np.inf, -np.inf], np.nan).fillna(0).sort_values("timestamp").reset_index(drop=True)
    X = pd.DataFrame({
        "Flow Duration": raw.flow_duration * US, "Total Fwd Packets": raw.tot_fwd_pkts,
        "Total Backward Packets": raw.tot_bwd_pkts, "Total Length of Fwd Packets": raw.totlen_fwd_pkts,
        "Total Length of Bwd Packets": raw.totlen_bwd_pkts, "Fwd Packet Length Mean": raw.fwd_pkt_len_mean,
        "Bwd Packet Length Max": raw.bwd_pkt_len_max, "Bwd Packet Length Mean": raw.bwd_pkt_len_mean,
        "Flow Bytes/s": raw.flow_byts_s, "Flow Packets/s": raw.flow_pkts_s, "Bwd Packets/s": raw.bwd_pkts_s,
        "Flow IAT Mean": raw.flow_iat_mean * US, "Flow IAT Std": raw.flow_iat_std * US,
        "Down/Up Ratio": raw.tot_bwd_pkts / raw.tot_fwd_pkts.replace(0, 1),
        "Average Packet Size": raw.pkt_size_avg, "Packet Length Std": raw.pkt_len_std,
        "Active Mean": raw.active_mean * US, "Idle Mean": raw.idle_mean * US,
    })
    X["Bwd_Fwd_Byte_Ratio"] = X["Total Length of Bwd Packets"] / (X["Total Length of Fwd Packets"] + 1)
    X = X[FEATS]
    Z = scaler.transform(np.log1p(X.clip(lower=0).values))
    raw["if_flag"]  = model.predict(Z) == -1                 # Layer 1
    raw["if_score"] = model.decision_function(Z)
    raw["MB_out"]   = X["Total Length of Bwd Packets"] / MB
    raw["MB_in"]    = X["Total Length of Fwd Packets"] / MB
    raw["ratio"]    = X["Bwd_Fwd_Byte_Ratio"]
    shape = (raw.MB_out >= 1) & (raw.ratio >= 10)            # Layer 2
    raw["verdict"] = np.where(raw.if_flag & shape, "EXFIL", np.where(raw.if_flag, "ANOMALY-OTHER", "NORMAL"))
    return raw

## 3. Round 1: six tools, 50 MB file
smbclient, impacket-smbclient, smbmap, crackmapexec, rpcclient, Metasploit smb_enumshares.
Ground truth: a flow that moved >= 10 MB to the attacker is a real exfiltration (impacket's `-c` syntax was invalid, so its
downloads did not occur; those sessions count as non-exfil).

In [3]:
r1 = score(f"{BASE}/smb_flows_v3.csv")
r1["truth"] = np.where(r1.MB_out >= 10, "exfil", "non-exfil")
print(f"Flows: {len(r1)}\n")
print(pd.crosstab(r1["truth"], r1["verdict"], margins=True))
e, n = r1[r1.truth == "exfil"], r1[r1.truth == "non-exfil"]
print(f"\nExfil labelled EXFIL     : {(e.verdict=='EXFIL').sum()}/{len(e)}")
print(f"Non-exfil labelled EXFIL : {(n.verdict=='EXFIL').sum()}/{len(n)}")

Flows: 81

verdict    EXFIL  NORMAL  All
truth                        
exfil         20       0   20
non-exfil      0      61   61
All           20      61   81

Exfil labelled EXFIL     : 20/20
Non-exfil labelled EXFIL : 0/61


## 4. Round 2: varied exfiltration (1 to 100 MB, bulk small files) and a normal macOS user

In [4]:
ex = score(f"{BASE}/r2_exfil.csv")
bn = score(f"{BASE}/r2_benign.csv")
ex["kind"] = pd.cut(ex.MB_out, [-1, 0.5, 2, 4, 10, 50, 1e9],
                    labels=["attacker-noise", "1MB file", "mget 30 small", "5MB file", "20MB file", "100MB file"])
print("=== Exfiltration by transfer type ===")
print(ex.groupby("kind", observed=True).agg(flows=("verdict", "size"), caught_IF=("if_flag", "sum"),
      labelled_EXFIL=("verdict", lambda v: (v == "EXFIL").sum()),
      min_score=("if_score", "min"), max_score=("if_score", "max")).round(3).to_string())
real = ex[ex.kind != "attacker-noise"]
print(f"\nExfil sessions labelled EXFIL : {(real.verdict=='EXFIL').sum()}/{len(real)}")
print(f"Attacker noise labelled EXFIL : {(ex[ex.kind=='attacker-noise'].verdict=='EXFIL').sum()}/{(ex.kind=='attacker-noise').sum()}")
print(f"\n=== macOS user (browse, read, upload, occasional download) ===")
print(f"Flagged by Layer 1: {bn.if_flag.sum()}/{len(bn)} | labelled EXFIL: {(bn.verdict=='EXFIL').sum()}/{len(bn)}")
print(bn[cols].round(3).to_string())

=== Exfiltration by transfer type ===
                flows  caught_IF  labelled_EXFIL  min_score  max_score
kind                                                                  
attacker-noise      8          0               0      0.061      0.063
1MB file            5          5               5     -0.034     -0.020
mget 30 small       3          3               3     -0.040     -0.039
5MB file            5          5               5     -0.039     -0.033
20MB file           8          8               8     -0.046     -0.042
100MB file          5          5               5     -0.053     -0.044

Exfil sessions labelled EXFIL : 26/26
Attacker noise labelled EXFIL : 0/8

=== macOS user (browse, read, upload, occasional download) ===
Flagged by Layer 1: 5/10 | labelled EXFIL: 0/10
             timestamp          src_ip  src_port  flow_duration  MB_out  MB_in  ratio  if_score        verdict
0  2026-11-25 10:02:49  192.168.10.114     53052          0.138   0.000  0.000  0.394     0.102 

## 5. Round 3: legitimate download-only user
Tests the known limitation: at flow level a legitimate download looks the same as exfiltration of the same size.

In [5]:
dl = score(f"{BASE}/r3_benign_dl.csv")
print(f"Download-only user: flagged by Layer 1 {dl.if_flag.sum()}/{len(dl)}, labelled EXFIL {(dl.verdict=='EXFIL').sum()}/{len(dl)}")
print(dl[cols].round(3).to_string())

Download-only user: flagged by Layer 1 5/10, labelled EXFIL 5/10
             timestamp          src_ip  src_port  flow_duration  MB_out  MB_in   ratio  if_score verdict
0  2026-11-25 10:20:53  192.168.10.114     53119          0.139   0.000  0.000   0.394     0.102  NORMAL
1  2026-11-25 10:20:55  192.168.10.114     53122         17.930  27.250  0.348  78.280    -0.031   EXFIL
2  2026-11-25 10:21:18  192.168.10.114     53123          0.153   0.000  0.000   0.580     0.107  NORMAL
3  2026-11-25 10:21:18  192.168.10.114     53125         15.447  27.248  0.524  51.965    -0.031   EXFIL
4  2026-11-25 10:21:39  192.168.10.114     53126          0.152   0.000  0.000   0.394     0.101  NORMAL
5  2026-11-25 10:21:39  192.168.10.114     53128         13.823  27.250  0.328  83.054    -0.033   EXFIL
6  2026-11-25 10:21:57  192.168.10.114     53129          0.139   0.000  0.000   0.394     0.102  NORMAL
7  2026-11-25 10:21:57  192.168.10.114     53131         17.415  27.249  0.341  79.955    -0.03

## 6. Layer 3: per-host context
Learning period = the Mac's normal activity (round 2). Baseline = its largest 10-minute outbound volume.

In [6]:
hist  = bn
known = set(hist.src_ip)

def win_sum(df, minutes=10):
    df = df.assign(t=pd.to_datetime(df.timestamp)).sort_values("t")
    out = pd.Series(0.0, index=df.index)
    for ip, g in df.groupby("src_ip"):
        out.loc[g.index] = g.set_index("t")["MB_out"].rolling(f"{minutes}min").sum().values
    return df.assign(win_MB=out)

base = win_sum(hist).groupby("src_ip")["win_MB"].max().to_dict()
print("Learned baseline (max MB per 10 min):", {k: round(v, 1) for k, v in base.items()})

ev = win_sum(pd.concat([r1.assign(set="R1 attack (Kali)"), ex.assign(set="R2 attack (Kali)"),
                        dl.assign(set="R3 legit download (Mac)")], ignore_index=True))

def layer3(r):
    if r.verdict != "EXFIL":          return "none"
    if r.src_ip not in known:         return "HIGH (unknown host)"
    if r.win_MB > 3 * base[r.src_ip]: return "MEDIUM (above own baseline)"
    return "LOW (within own baseline)"

ev["alert"] = ev.apply(layer3, axis=1)
print("\n=== Final alert level by traffic set ===")
print(pd.crosstab(ev["set"], ev["alert"], margins=True))
print("\nMac legit-download sessions:")
print(ev[ev.set.str.startswith("R3") & (ev.verdict == "EXFIL")][["timestamp", "MB_out", "win_MB", "alert"]].round(1).to_string())
ev.to_csv(f"{BASE}/all_flows_layer3.csv", index=False)

Learned baseline (max MB per 10 min): {'192.168.10.114': 34.5}

=== Final alert level by traffic set ===
alert                    HIGH (unknown host)  LOW (within own baseline)  MEDIUM (above own baseline)  none  All
set                                                                                                            
R1 attack (Kali)                          20                          0                            0    61   81
R2 attack (Kali)                          26                          0                            0     8   34
R3 legit download (Mac)                    0                          3                            2     5   10
All                                       46                          3                            2    74  125

Mac legit-download sessions:
               timestamp  MB_out  win_MB                        alert
116  2026-11-25 10:20:55    27.2    27.2    LOW (within own baseline)
118  2026-11-25 10:21:18    27.2    54.5    LOW (with

## 7. Results and limitations

**Results (lab, 2026-11-25):** all 46 exfiltration sessions (1-100 MB, bulk small files, smbclient / smbmap / impacket)
raised HIGH alerts; 0 of 69 attacker recon / noise flows raised any alert; legitimate downloads stayed LOW until the
host exceeded 3x its own volume baseline (MEDIUM), never HIGH. Layer 1 false positive rate on unseen CICIDS days: 0.8%.

**Limitations:**
- Layers 1-2 detect *large one-directional transfers*; they cannot by themselves tell theft from a legitimate download.
- In the lab, HIGH comes from the attacker being a new host. An insider on a known machine is caught only by the
  volume rule, whose baseline here is minutes of history; production needs weeks.
- CICIDS-2017 provides only 1,932 benign port-445 flows from one network. Next step: CSE-CIC-IDS2018 for a larger,
  cross-dataset benign test.
- Test files are random data; attack tool set is limited to four tools that moved data.